### Data Loading

In [1]:
import pandas as pd

#load cleaned data
buzzfeed_real = pd.read_pickle('data/buzzfeed_real_clean.pkl')
buzzfeed_fake = pd.read_pickle('data/buzzfeed_fake_clean.pkl')

misinfo_real = pd.read_pickle('data/misinfo_real_clean.pkl')
misinfo_fake = pd.read_pickle('data/misinfo_fake_clean.pkl')

#cut out a large portion of the misinfo data to resolve memory issues
misinfo_real = misinfo_real[:2500]
misinfo_fake = misinfo_fake[:2500]

In [2]:
buzzfeed_real.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ..."


In [3]:
buzzfeed_fake.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ..."


In [4]:
misinfo_real.head(1)

,text,real_label
0,The head of a conservative Republican faction ...,1


In [5]:
misinfo_fake.head(1)

,text,real_label
0,Donald Trump just couldn t wish all Americans ...,0


### Preprocessing

In [6]:
#function to be applied to article titles and contents for initial text cleaning
import re
import string

def clean_text(text):
    
    #type checking
    if not isinstance(text, str):
        return []

    #lowercasing
    text = text.lower()
    
    #removing punctuation
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\w*\d\w*', '', text)

    text = re.sub('[’‘’“”…]', '', text)
    text = re.sub('\n', '', text)
    text = ' '.join(re.findall(r'\b[a-zA-Z0-9]+\b', text))


    return text

<>:15: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\w'
<>:15: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\w'
C:\Users\kmbbm\AppData\Local\Temp\ipykernel_23468\2203787657.py:15: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
C:\Users\kmbbm\AppData\Local\Temp\ipykernel_23468\2203787657.py:17: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [7]:
#fake news preprocessing
buzzfeed_fake['clean_title'] = buzzfeed_fake['title'].apply(clean_text)
buzzfeed_fake['clean_text'] = buzzfeed_fake['text'].apply(clean_text)

#real news preprocessing
buzzfeed_real['clean_title'] = buzzfeed_real['title'].apply(clean_text)
buzzfeed_real['clean_text'] = buzzfeed_real['text'].apply(clean_text)

#misinfo dataset preprocessing
misinfo_fake['clean_text'] = misinfo_fake['text'].astype(str).apply(clean_text)
misinfo_real['clean_text'] = misinfo_real['text'].astype(str).apply(clean_text)

In [8]:
buzzfeed_fake.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data,clean_title,clean_text
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ...",proof the mainstream media is manipulating the...,i woke up this morning to find a variation of ...


In [9]:
buzzfeed_real.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data,clean_title,clean_text
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ...",proof the mainstream media is manipulating the...,i woke up this morning to find a variation of ...


In [10]:
misinfo_real.head(1)

,text,real_label,clean_text
0,The head of a conservative Republican faction ...,1,the head of a conservative republican faction ...


In [11]:
misinfo_fake.head(1)

,text,real_label,clean_text
0,Donald Trump just couldn t wish all Americans ...,0,donald trump just couldn t wish all americans ...


In [12]:
#set up data for train test split

#add labels to buzzfeed data
buzzfeed_fake['real_label'] = 0
buzzfeed_real['real_label'] = 1

#create single dataframe and shuffle it up
full_df = pd.concat([buzzfeed_fake, buzzfeed_real, misinfo_fake, misinfo_real], axis=0, ignore_index = True).sample(frac = 1, random_state = 101).reset_index(drop = True)

In [13]:
#ensure all text is strings
full_df['clean_text'] = full_df['clean_text'].astype(str)

### Tokenizing

In [14]:
#create a document term matrix
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words = 'english')
df_cv = cv.fit_transform(full_df['clean_text'])

df_dtm = pd.DataFrame(df_cv.toarray(), columns = cv.get_feature_names_out(), index = full_df.index)

df_dtm

,aaa,aafter,aaliyah,aaplo,aaplus,aaron,aaroncovfefe,aaronshhh,aarp,ab,...,zuckett,zuker,zukunft,zuppello,zyklon,zypries,zz,zztaine,zzzzaaaacccchhh,zzzzzzzzzzzzz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5177,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5178,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5179,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5180,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
#append labels to count matrix
labelled_dtm = pd.concat([df_dtm, full_df['real_label']], axis = 1)

In [16]:
#save pkl for model training
pd.to_pickle(labelled_dtm, 'data/labelled_dtm.pkl')

### Tokenizing with Context

In [17]:
#use text frequency inverse document frequency to create a document term matrix
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
df_tfidf = tfidf.fit_transform(full_df['clean_text'])

df_tfidf_dtm = pd.DataFrame(df_tfidf.toarray(), columns=tfidf.get_feature_names_out(), index=full_df.index)
labelled_tfidf_dtm = pd.concat([df_tfidf_dtm, full_df['real_label']], axis=1)

labelled_tfidf_dtm.head()

,aaa,aafter,aaliyah,aaplo,aaplus,aaron,aaroncovfefe,aaronshhh,aarp,ab,...,zuker,zukunft,zuppello,zyklon,zypries,zz,zztaine,zzzzaaaacccchhh,zzzzzzzzzzzzz,real_label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [18]:
labelled_tfidf_dtm.shape

(5182, 53541)

In [19]:
#pickle for training later
pd.to_pickle(labelled_tfidf_dtm, 'data/labelled_tfidf_dtm.pkl')

### Saving Trained Vectorizers

In [20]:
#save count vectorizer and tfidf vectorizer for later use
import joblib
joblib.dump(cv, 'data/cv.pkl')
joblib.dump(tfidf, 'data/tfidf.pkl')

['data/tfidf.pkl']